# Inference Multi-Character Aksara Lontara
## Fine-Tuned BLIP – Deteksi 1 atau 2 Karakter per Gambar

### Penjelasan Pendekatan

Model BLIP yang telah dilatih sebelumnya dilatih pada gambar **tunggal** (1 karakter per gambar).
Oleh karena itu, jika gambar input mengandung **2 karakter sekaligus**, kita perlu strategi khusus agar model tetap bisa mengenali masing-masing karakter dengan akurat.

**Alur kerja notebook ini:**

1. **Segmentasi Karakter** – Menggunakan OpenCV untuk mendeteksi dan memisahkan area tiap karakter dalam gambar (dari kiri ke kanan).
2. **Inference Individual** – Setiap potongan karakter di-resize ke 384×384 dan diberikan ke model BLIP satu per satu.
3. **Penggabungan Caption** – Hasil caption dari tiap karakter digabungkan menjadi satu deskripsi utuh yang menjelaskan kedua karakter.

**Catatan:**
- Jika gambar hanya mengandung **1 karakter**, notebook akan langsung menghasilkan caption normal (tidak melakukan segmentasi).
- Jika gambar mengandung **2 karakter**, masing-masing akan di-caption secara terpisah lalu digabungkan.
- Pastikan karakter dalam gambar tidak saling tumpang tindih dan memiliki kontras cukup dengan background agar segmentasi berhasil.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: SETUP, IMPORT & MOUNT GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════

import os
import re
import json
import cv2
import math
import torch
import numpy as np
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
import matplotlib.pyplot as plt

# Mount Google Drive (khusus Colab)
from google.colab import drive
drive.mount('/content/drive')

# ─── Konfigurasi ───
DRIVE_DIR  = '/content/drive/MyDrive/ImageCaptioning'
MODEL_DIR  = f'{DRIVE_DIR}/model/blip_lontara/best_model'
TEST_DIR   = f'{DRIVE_DIR}/datatest'           # Folder gambar uji
MAX_LENGTH = 64
NUM_BEAMS  = 4
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device aktif: {DEVICE}')
print(f'Model path  : {MODEL_DIR}')
print(f'Test path   : {TEST_DIR}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: LOAD MODEL BLIP HASIL FINE-TUNING
# ═══════════════════════════════════════════════════════════════

assert os.path.exists(MODEL_DIR), f'Model tidak ditemukan di {MODEL_DIR}! Pastikan hasil training sudah di-copy ke Drive.'

print(f'Memuat model dari: {MODEL_DIR}')
processor = BlipProcessor.from_pretrained(MODEL_DIR)
model     = BlipForConditionalGeneration.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

print('✅ Model BLIP siap digunakan!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: FUNGSI SEGMENTASI & INFERENCE MULTI-KARAKTER
# ═══════════════════════════════════════════════════════════════

def segment_characters(image_path, min_area=200, padding=15):
    """
    Deteksi dan memisahkan karakter dalam gambar menggunakan OpenCV contour.
    Mengembalikan list bounding-box (x1, y1, x2, y2) terurut dari kiri ke kanan.
    """
    img_cv = cv2.imread(image_path)
    if img_cv is None:
        raise ValueError(f'Gagal membaca gambar: {image_path}')

    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    # Otsu threshold → binary inverted (karakter = putih, bg = hitam)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Bersihkan noise kecil
    kernel = np.ones((3, 3), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h_img, w_img = gray.shape
    boxes = []
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = cv2.contourArea(cnt)
        # Filter: area cukup besar, bukan garis tipis, bukan full image
        if (area >= min_area and
            15 < w < w_img * 0.9 and
            15 < h < h_img * 0.9):
            x1 = max(0, x - padding)
            y1 = max(0, y - padding)
            x2 = min(w_img, x + w + padding)
            y2 = min(h_img, y + h + padding)
            boxes.append((x1, y1, x2, y2))

    # Urutkan dari kiri ke kanan (untuk bacaan Lontara)
    boxes = sorted(boxes, key=lambda b: b[0])
    return boxes, img_cv


def crop_character(pil_img, box, target_size=(384, 384)):
    """Crop sesuai bounding-box lalu resize ke ukuran yang diminta model."""
    x1, y1, x2, y2 = box
    cropped = pil_img.crop((x1, y1, x2, y2))
    return cropped.resize(target_size, Image.LANCZOS)


def predict_single(pil_img):
    """Captioning 1 gambar (1 karakter) menggunakan BLIP."""
    inputs = processor(images=pil_img, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=2
        )
    return processor.decode(output[0], skip_special_tokens=True)


def extract_char_name(caption):
    """Mencoba mengekstrak nama karakter dari caption untuk intro gabungan."""
    patterns = [
        r'karakter lontara[^.]*?yaitu karakter (\w+)',
        r'karakter (\w+)[\.,\s]',
        r'adalah (\w+)[\.,\s]',
    ]
    for pat in patterns:
        m = re.search(pat, caption)
        if m:
            return m.group(1)
    return None


def combine_captions(captions):
    """
    Gabungkan caption individual menjadi satu deskripsi utuh.
    Jika 1 karakter → kembalikan apa adanya.
    Jika 2 karakter → buat intro gabungan lalu detail per karakter.
    """
    n = len(captions)
    if n == 0:
        return 'Tidak ada karakter yang terdeteksi.'
    if n == 1:
        return captions[0]

    # Ekstrak nama karakter untuk intro
    names = [extract_char_name(c) for c in captions]
    if all(names):
        intro = f'Gambar ini terdiri dari {n} karakter lontara yaitu karakter {" dan ".join(names)}.'
    else:
        intro = f'Gambar ini terdiri dari {n} karakter lontara.'

    # Bersihkan caption individual agar tidak berulang
    details = []
    for i, cap in enumerate(captions, 1):
        # Hapus intro redundan
        clean = re.sub(r'^Gambar ini terdiri dari \d+ karakter lontara[^.]*\.', '', cap).strip()
        # Ganti subjek agar jelas ke-{i}
        clean = re.sub(r'^Karakter lontara pada gambar ini adalah (\w+)',
                       f'Karakter ke-{i} (\1)', clean)
        if clean:
            details.append(clean)

    return intro + ' ' + ' '.join(details)


def predict_multi(image_path):
    """
    Entry point utama.
    Menerima path gambar → segmentasi → inference per karakter → caption gabungan.
    Mengembalikan: (caption_gabungan, list_img_per_karakter, list_caption_per_karakter, list_boxes)
    """
    pil_img = Image.open(image_path).convert('RGB')
    w_img, h_img = pil_img.size

    # Coba segmentasi
    boxes, _ = segment_characters(image_path)

    # Filter: jika box tunggal yang hampir full gambar, anggap 1 karakter
    img_area = w_img * h_img
    valid_boxes = [b for b in boxes if ((b[2]-b[0])*(b[3]-b[1])) < img_area * 0.95]

    # Jika tidak ada/tidak valid → fallback single inference
    if len(valid_boxes) <= 1:
        cap = predict_single(pil_img)
        return cap, [pil_img], [cap], valid_boxes

    # Inference per karakter
    char_imgs = []
    char_caps = []
    for box in valid_boxes:
        cimg = crop_character(pil_img, box)
        char_imgs.append(cimg)
        char_caps.append(predict_single(cimg))

    combined = combine_captions(char_caps)
    return combined, char_imgs, char_caps, valid_boxes

print('✅ Fungsi segmentasi & multi-character inference siap!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: DEMO INFERENCE (SATU GAMBAR)
# ═══════════════════════════════════════════════════════════════

# Ganti dengan nama file gambar uji Anda (1 atau 2 karakter)
TEST_IMAGE = f'{TEST_DIR}/contoh_2_karakter.png'   # <-- sesuaikan nama file

if not os.path.exists(TEST_IMAGE):
    print(f'⚠️ Gambar tidak ditemukan: {TEST_IMAGE}')
    print('Pastikan Anda sudah meng-upload gambar ke folder datatest di Google Drive.')
else:
    combined, char_imgs, char_caps, boxes = predict_multi(TEST_IMAGE)

    print('=' * 60)
    print(f'Jumlah karakter terdeteksi: {len(char_imgs)}')
    print('=' * 60)
    print('📌 CAPTION GABUNGAN:')
    print(combined)
    print('=' * 60)
    print('📋 DETAIL PER KARAKTER:')
    for i, cap in enumerate(char_caps, 1):
        print(f'  [{i}] {cap}')

    # ─── Visualisasi ───
    n = len(char_imgs)
    if n == 1:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(Image.open(TEST_IMAGE).convert('RGB'))
        axes[0].set_title('Gambar Input')
        axes[0].axis('off')
        axes[1].imshow(char_imgs[0])
        axes[1].set_title('Hasil Caption', fontsize=9)
        axes[1].axis('off')
    else:
        fig, axes = plt.subplots(2, n + 1, figsize=(4.5 * (n + 1), 9))

        # Baris 1: Input + tiap karakter hasil crop
        axes[0, 0].imshow(Image.open(TEST_IMAGE).convert('RGB'))
        axes[0, 0].set_title('Gambar Input')
        axes[0, 0].axis('off')
        for i, img in enumerate(char_imgs):
            axes[0, i + 1].imshow(img)
            axes[0, i + 1].set_title(f'Karakter {i+1}')
            axes[0, i + 1].axis('off')

        # Baris 2: Caption gabungan + caption individual
        axes[1, 0].text(0.5, 0.5, combined, wrap=True,
                        horizontalalignment='center', verticalalignment='center',
                        fontsize=10, transform=axes[1, 0].transAxes)
        axes[1, 0].set_title('Caption Gabungan')
        axes[1, 0].axis('off')
        for i, cap in enumerate(char_caps):
            axes[1, i + 1].text(0.5, 0.5, cap, wrap=True,
                               horizontalalignment='center', verticalalignment='center',
                               fontsize=9, transform=axes[1, i + 1].transAxes)
            axes[1, i + 1].set_title(f'Caption Karakter {i+1}')
            axes[1, i + 1].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: BATCH INFERENCE (SELURUH FOLDER DATATEST)
# ═══════════════════════════════════════════════════════════════

VALID_EXT = ('.png', '.jpg', '.jpeg')

if not os.path.exists(TEST_DIR):
    print(f'⚠️ Folder {TEST_DIR} tidak ditemukan. Buat folder datatest di Google Drive Anda.')
else:
    images = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(VALID_EXT)])
    print(f'Ditemukan {len(images)} gambar di folder datatest\n')

    results = []
    for img_name in images:
        img_path = os.path.join(TEST_DIR, img_name)
        try:
            cap, _, _, _ = predict_multi(img_path)
            results.append({'gambar': img_name, 'caption': cap})
            print(f'{img_name} → {cap}')
        except Exception as e:
            print(f'{img_name} → ERROR: {e}')

    # Simpan hasil ke Drive
    out_dir = f'{DRIVE_DIR}/hasil_inference_multi'
    os.makedirs(out_dir, exist_ok=True)
    out_file = f'{out_dir}/hasil_multi.json'
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f'\n✅ Hasil disimpan di: {out_file}')